In [1]:

"""
Project: Trading Bot Using Indicators of choice

Goal:
The goal of this project is to build a trading bot that will show good opportunites to buy stock based on certain indicators. The indicators
chosen are : Stochastic RSI, RSI (Relative Strength Index), Moving Average (200 day). These will be viewed from the weekly chart. 
With this tool we would be able to scrape stocks for potential setups that mirror what I would look for in a trade setup.
The strategy will then be backtested to show its success rate. The trading bot itself will then point out if a stock is 
a "buy" depending on whether indicators are showing the correct parameters

Features:
- Stock indicator check
- Backtesting tool
- Multi-stock scanner

Status: Working on inputing the strategy (based on my actual trading strategy)
"""

import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
# Strategy Settings

'''
Strategy Rules (all of these are on the weekly timeframe):

Buy the stock if:
- RSI > 30
- StochRSI < 20 (both K and D lines)
- price is > 200 day moving average

Otherwise, the stock is not a Buy
'''
# Thresholds for the indicators

RSI_Threshold = 30
SMA_Period = 200
StochRSI_Threshold = 20

In [77]:
# Stock(s) that will be used to test
ticker = "AAPL"

df = yf.download(ticker, period='5y', interval='1wk')

[*********************100%***********************]  1 of 1 completed


In [34]:
# Indicator functions

#RSI Formula uses Wilder smoothing, math to make it similar to TradingView's RSI
def calculate_rsi(close, period=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

# K and D periods for RSI both resemble those of TradingView
def calculate_stoch_rsi(close, rsi_period=14, stoch_period=14, k_period=3, d_period=3):
    rsi = calculate_rsi(close, rsi_period)

    min_rsi = rsi.rolling(stoch_period).min()
    max_rsi = rsi.rolling(stoch_period).max()

    stoch_rsi = (rsi - min_rsi) / (max_rsi - min_rsi)

    k = stoch_rsi.rolling(k_period).mean()
    d = k.rolling(d_period).mean()

    return {'k': k, 'd': d}


def calculate_sma(close, period=200):
    sma = close.rolling(window=period).mean()

    return sma
    



In [90]:
# Signal Function

# closing price of the stock and returning the last RSI
close = df['Close']['AAPL']

# calling the RSI
rsi = calculate_rsi(close, 14)
df['RSI'] = rsi

last_rsi = rsi.iloc[-1]

print(f'RSI: {last_rsi:.2f}') # Using :.2f to limit results to 2 decimal places. Too many numbers otherwise

if last_rsi < 20:
    print('This stock is way oversold')
elif last_rsi > 20 and last_rsi < 50:
    print('Stock is leaning oversold')
elif last_rsi < 70 and last_rsi >= 50:
    print('Stock is leaning overbought')
elif last_rsi >= 70:
    print('This stock is overbought for sure.')


# calling the StochRSI (k and d lines)

stoch_rsi = calculate_stoch_rsi(close)

print(f"Stoch_RSI: K={stoch_rsi['k'].iloc[-1]:.2f}, D={stoch_rsi['d'].iloc[-1]:.2f}")

stoch_k = stoch_rsi['k'].iloc[-1]
stoch_d = stoch_rsi['d'].iloc[-1]

if stoch_k <= .20 and stoch_d.iloc[-1] <= .20:
    print('Momentum is bearish')
elif stoch_k >= .70 and stoch_d >= .70:
    print('Full bullish momentum')
elif stoch_k >= .70 and stoch_d <= .70: # k line can lead before d fully confirms the bullishness
    print('StochRSI is showing bullish momentum')
else:
    print('StochRSI in midrange, evaluate further')

# calling the 200SMA 

sma_real = calculate_sma(close)

print(f'SMA(200): {sma_real.iloc[-1]:.2f}')

if close.iloc[-1] >= sma_real.iloc[-1]:
    print('This stock is bull market bullish')
else:
    print('Stock may be in a bear trend.')


RSI: 67.31
Stock is leaning overbought
Stoch_RSI: K=0.91, D=0.78
Full bullish momentum
SMA(200): 201.41
This stock is bull market bullish


In [6]:
# Some test scripts here


In [8]:
# Signal Ticker Test

In [9]:
# Backtester

In [10]:
# Multi-Stock Scanner

In [11]:
# Results / Charts